# EXEMPLO — Pipeline Cronológico de Testagem Comparativa de Embeddings para RAG Jurídico

**Curso:** MBA RAG & CAG Aplicados a Direito e Segurança Pública
**Aula:** 1 — Fundamentos
**Tipo:** Exemplo demonstrativo (atividade prática integradora)
**Duração estimada:** 90 minutos
**Pré-requisitos:** LAB1 (ambiente) + LAB2 (embeddings + avaliação) concluídos

---

## 🎯 O que este pipeline faz

Você vai executar, **passo a passo cronologicamente**, uma testagem comparativa de **4 modelos de embedding** para identificar qual é o mais adequado para sistemas RAG jurídicos.

| # | Modelo | Origem | Dim | Custo |
|---|--------|--------|-----|-------|
| 1 | `nomic-embed-text` | Ollama (API REST) | 768 | Gratuito (local) |
| 2 | `mxbai-embed-large` | Ollama (API REST) | 1024 | Gratuito (local) |
| 3 | `BAAI/bge-m3` | sentence-transformers (Python direto) | 1024 | Gratuito (local) |
| 4 | `paraphrase-multilingual-mpnet-base-v2` | sentence-transformers | 768 | Gratuito (local) |

## 🗺️ Fluxo cronológico (6 fases)

```
Fase 0: Preparação ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ verificar Ollama, LangFuse, GPU
   ↓
Fase 1: Corpus + Ground Truth ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10 docs jurídicos + 8 queries
   ↓
Fase 2: Encoding ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4 modelos × 10 docs (com timing)
   ↓
Fase 3: Avaliação Quantitativa ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Hit@K, MRR, NDCG@K, AUC-ROC
   ↓
Fase 4: RAG End-to-End ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ retrieval + llama3.2:3b
   ↓
Fase 5: LangFuse ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ trace estruturado por modelo
   ↓
Fase 6: Parecer Técnico ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ recomendação final justificada
```

## 📊 No final do exemplo, você terá

- ✅ Tabela comparativa multimétrica dos 4 modelos
- ✅ Radar chart visual mostrando forças/fraquezas
- ✅ Resposta RAG completa para cada modelo (comparativa)
- ✅ Traces LangFuse navegáveis com latência, tokens e qualidade
- ✅ **Parecer técnico fundamentado** com recomendação por cenário de uso jurídico

> **Robustez:** o notebook funciona mesmo sem Ollama (usa só locais), sem LangFuse (pula tracing) e sem GPU (cai para CPU). Cada fase verifica seus pré-requisitos antes de executar.


---

# 🔧 FASE 0 — Preparação do Ambiente

Verificações cronológicas:
1. Python / PyTorch / GPU
2. Ollama UP + modelos disponíveis
3. LangFuse UP (opcional)
4. Imports e configuração global


In [1]:
# PASSO 0.1 — Verificar Python, PyTorch, GPU
import sys, platform

print('═' * 60)
print('AMBIENTE PYTHON')
print('═' * 60)
print(f'  Python:       {sys.version.split()[0]}')
print(f'  Plataforma:   {platform.system()} {platform.release()}')

try:
    import torch
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'  PyTorch:      {torch.__version__}')
    print(f'  Dispositivo:  {DEVICE}')
    if DEVICE == 'cuda':
        print(f'  GPU:          {torch.cuda.get_device_name(0)}')
        print(f'  VRAM total:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
except ImportError:
    DEVICE = 'cpu'
    print('  PyTorch: não instalado — usando CPU')


════════════════════════════════════════════════════════════
AMBIENTE PYTHON
════════════════════════════════════════════════════════════
  Python:       3.12.7
  Plataforma:   Windows 11
  PyTorch:      2.12.0+cpu
  Dispositivo:  cpu


In [3]:
# PASSO 0.2 — Verificar Ollama UP e modelos disponíveis
import requests, os

OLLAMA_BASE_URL = os.environ.get('OLLAMA_BASE_URL', 'http://localhost:11434')

ollama_status = {'online': False, 'modelos': [], 'tem_llm': False,
                 'tem_nomic': False, 'tem_mxbai': False}

try:
    r = requests.get(f'{OLLAMA_BASE_URL}/api/tags', timeout=5)
    r.raise_for_status()
    ollama_status['modelos'] = [m['name'] for m in r.json().get('models', [])]
    ollama_status['online'] = True
    ollama_status['tem_llm']   = any('llama' in m for m in ollama_status['modelos'])
    ollama_status['tem_nomic'] = any('nomic' in m for m in ollama_status['modelos'])
    ollama_status['tem_mxbai'] = any('mxbai' in m for m in ollama_status['modelos'])
except Exception as e:
    print(f'⚠️  Ollama OFFLINE: {e}')
    print('    Execute: ollama serve')

print('═' * 60)
print('OLLAMA')
print('═' * 60)
print(f'  Status:           {"🟢 ONLINE" if ollama_status["online"] else "🔴 OFFLINE"}')
print(f'  Modelos:          {ollama_status["modelos"]}')
print(f'  LLM disponível:   {"✅" if ollama_status["tem_llm"]   else "❌"} (precisa: llama3.2:3b)')
print(f'  nomic-embed-text: {"✅" if ollama_status["tem_nomic"] else "❌"}')
print(f'  mxbai-embed-large:{"✅" if ollama_status["tem_mxbai"] else "❌ (opcional — instale com: ollama pull mxbai-embed-large)"}')

if ollama_status['online']:
    if not ollama_status['tem_nomic']:
        print('\n  💡 Para usar Ollama no pipeline, execute:')
        print('     ollama pull nomic-embed-text')
        print('     ollama pull llama3.2:3b')


════════════════════════════════════════════════════════════
OLLAMA
════════════════════════════════════════════════════════════
  Status:           🟢 ONLINE
  Modelos:          ['paraphrase-multilingual:latest', 'nomic-embed-text:latest', 'bge-m3:latest']
  LLM disponível:   ❌ (precisa: llama3.2:3b)
  nomic-embed-text: ✅
  mxbai-embed-large:❌ (opcional — instale com: ollama pull mxbai-embed-large)


In [ ]:
# PASSO 0.3 — Verificar LangFuse (opcional)
from pathlib import Path

# Carrega .env do curso, se existir
try:
    from dotenv import load_dotenv
    for env_path in [Path.home() / 'mba-rag' / '.env', Path('.env'), Path('..') / '.env']:
        if env_path.exists():
            load_dotenv(env_path)
            print(f'📄 .env carregado: {env_path}')
            break
except ImportError:
    pass

LANGFUSE_PUBLIC_KEY = os.environ.get('LANGFUSE_PUBLIC_KEY', '')
LANGFUSE_SECRET_KEY = os.environ.get('LANGFUSE_SECRET_KEY', '')
LANGFUSE_HOST       = os.environ.get('LANGFUSE_HOST', 'http://localhost:3000')

LANGFUSE_OK = (
    LANGFUSE_PUBLIC_KEY
    and LANGFUSE_SECRET_KEY
    and not LANGFUSE_PUBLIC_KEY.endswith('_AQUI')
    and len(LANGFUSE_PUBLIC_KEY) > 10
)

print('═' * 60)
print('LANGFUSE')
print('═' * 60)
print(f'  Status:  {"🟢 CONFIGURADO" if LANGFUSE_OK else "⚪ NÃO CONFIGURADO (opcional)"}')
print(f'  Host:    {LANGFUSE_HOST}')
if LANGFUSE_OK:
    print(f'  PK:      {LANGFUSE_PUBLIC_KEY[:10]}...')
else:
    print('  Para ativar: adicione LANGFUSE_PUBLIC_KEY e LANGFUSE_SECRET_KEY ao ~/mba-rag/.env')
    print('  O pipeline continuará executando — apenas pulará a fase de tracing.')


In [ ]:
# PASSO 0.4 — Imports e configuração global
%pip install sentence-transformers FlagEmbedding faiss-cpu openai langfuse python-dotenv scikit-learn matplotlib pandas numpy --quiet

import json, time, uuid
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ndcg_score, roc_auc_score
from sklearn.metrics.pairwise import cosine_similarity
import faiss
from openai import OpenAI

# Configurações globais
OLLAMA_LLM_MODEL = os.environ.get('OLLAMA_LLM_MODEL', 'llama3.2:3b')

# Cliente OpenAI-compatível apontando para o Ollama (para geração)
client_llm = OpenAI(
    base_url=f'{OLLAMA_BASE_URL}/v1',
    api_key='ollama'
)

# Paleta de cores para os 4 modelos (consistente em todos os gráficos)
CORES_MODELOS = {
    'nomic-embed-text (Ollama)':     '#3498DB',  # azul
    'mxbai-embed-large (Ollama)':    '#9B59B6',  # roxo
    'BGE-M3 (local)':                '#E74C3C',  # vermelho
    'MiniLM-Multilingual (local)':   '#F39C12',  # laranja
}

print('✅ Imports prontos')
print(f'   LLM para geração: {OLLAMA_LLM_MODEL}')
print(f'   Dispositivo:      {DEVICE}')


---

# 📚 FASE 1 — Carregar Corpus e Construir Ground Truth

Vamos usar o **corpus jurídico oficial da Aula 1** (10 documentos em 6 categorias) e construir um conjunto de queries com **categorias-alvo conhecidas** para avaliação quantitativa.

A construção segue o mesmo padrão metodológico do LAB2 (escala de relevância 0-2):

- Grau 2: doc da MESMA categoria que a query
- Grau 1: doc de categoria tematicamente próxima
- Grau 0: doc não relacionado


In [ ]:
# PASSO 1.1 — Carregar corpus jurídico

CORPUS_PATH = Path('../datasets/corpus_juridico_aula1.json')
if not CORPUS_PATH.exists():
    # Fallback para execução em outras pastas
    CORPUS_PATH = Path('aula1/datasets/corpus_juridico_aula1.json')

with open(CORPUS_PATH, 'r', encoding='utf-8') as f:
    corpus = json.load(f)

# Normalizar para o formato esperado: texto = titulo + conteudo
for doc in corpus:
    if 'texto' not in doc:
        doc['texto'] = doc.get('conteudo', '')

print(f'📚 Corpus carregado: {len(corpus)} documentos')
print()
for doc in corpus:
    print(f'  [{doc["id"]:>2}] {doc["categoria"]:25s} | {doc["titulo"][:55]}')

# Mapeamento de categorias do corpus
categorias_corpus = sorted(set(d['categoria'] for d in corpus))
print(f'\n📁 {len(categorias_corpus)} categorias: {categorias_corpus}')


In [ ]:
# PASSO 1.2 — Definir queries de avaliação com ground truth

# Afinidade temática entre categorias (para escala NDCG 0-2)
afinidade = {
    'Crimes Funcionais':    ['Investigação Criminal'],
    'Investigação Criminal':['Crimes Funcionais', 'Direito Processual'],
    'Direito Processual':   ['Investigação Criminal'],
    'Crimes Patrimoniais':  [],
    'Crimes Contra a Vida': ['Violência Doméstica'],
    'Violência Doméstica':  ['Crimes Contra a Vida'],
}

queries_avaliacao = [
    {'query': 'desvio de dinheiro público por funcionário do governo',          'categoria_alvo': 'Crimes Funcionais'},
    {'query': 'pedido de soltura por falta de fundamentação',                   'categoria_alvo': 'Direito Processual'},
    {'query': 'arrombamento de casa para roubo de bens',                        'categoria_alvo': 'Crimes Patrimoniais'},
    {'query': 'exame pericial em local de homicídio com análise balística',     'categoria_alvo': 'Crimes Contra a Vida'},
    {'query': 'investigação federal sobre organização criminosa de tráfico',    'categoria_alvo': 'Investigação Criminal'},
    {'query': 'agressão física contra mulher em ambiente familiar',             'categoria_alvo': 'Violência Doméstica'},
    {'query': 'golpe aplicado em idoso com promessa de prêmio',                 'categoria_alvo': 'Crimes Patrimoniais'},
    {'query': 'medida cautelar com tornozeleira eletrônica',                    'categoria_alvo': 'Direito Processual'},
]

for q in queries_avaliacao:
    q['categorias_proximas'] = afinidade.get(q['categoria_alvo'], [])

df_queries = pd.DataFrame(queries_avaliacao)
print(f'📋 Ground truth: {len(df_queries)} queries de avaliação')
print()
print(df_queries[['query', 'categoria_alvo']].to_string())


In [ ]:
# PASSO 1.3 — Construir matriz de relevância (n_queries × n_docs)

def grau_relevancia(cat_doc: str, query: dict) -> int:
    if cat_doc == query['categoria_alvo']:
        return 2
    if cat_doc in query['categorias_proximas']:
        return 1
    return 0

relevancias = np.zeros((len(queries_avaliacao), len(corpus)), dtype=int)
for i, q in enumerate(queries_avaliacao):
    for j, doc in enumerate(corpus):
        relevancias[i, j] = grau_relevancia(doc['categoria'], q)

print(f'📊 Matriz de relevâncias: shape = {relevancias.shape}')
print(f'   Docs grau 2 por query (média): {(relevancias == 2).sum(axis=1).mean():.1f}')
print(f'   Docs grau 1 por query (média): {(relevancias == 1).sum(axis=1).mean():.1f}')
print(f'   Docs grau 0 por query (média): {(relevancias == 0).sum(axis=1).mean():.1f}')

# Visualização rápida
fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(relevancias, cmap='RdYlGn', aspect='auto', vmin=0, vmax=2)
ax.set_yticks(range(len(queries_avaliacao)))
ax.set_yticklabels([f'q{i+1}' for i in range(len(queries_avaliacao))])
ax.set_xticks(range(len(corpus)))
ax.set_xticklabels([f'doc{d["id"]}' for d in corpus])
ax.set_xlabel('Documentos')
ax.set_ylabel('Queries')
ax.set_title('Matriz de Relevância (0=irrelev / 1=próxima / 2=mesma cat)')
plt.colorbar(im, ax=ax, fraction=0.025)
for i in range(relevancias.shape[0]):
    for j in range(relevancias.shape[1]):
        ax.text(j, i, str(relevancias[i, j]), ha='center', va='center',
                color='black', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()


---

# 🔢 FASE 2 — Encoding com os 4 Modelos

Vamos encodar **o mesmo corpus de 10 docs** com **4 modelos diferentes**, cronologicamente.

| Ordem | Modelo | Acesso | Dim |
|-------|--------|--------|-----|
| 1 | `nomic-embed-text` | Ollama HTTP | 768 |
| 2 | `mxbai-embed-large` | Ollama HTTP | 1024 |
| 3 | `BAAI/bge-m3` | Python direto (sentence-transformers) | 1024 |
| 4 | `paraphrase-multilingual-mpnet-base-v2` | Python direto | 768 |

**Modelos indisponíveis serão pulados graciosamente** — você pode rodar com 2, 3 ou 4 modelos conforme o ambiente.

Para cada modelo, mediremos:
- Tempo de encoding do corpus
- Tempo médio por documento
- Dimensionalidade efetiva


In [ ]:
# PASSO 2.1 — Funções de encoding (Ollama + local)

def encode_ollama(textos, modelo: str):
    """Encoda lista de textos via Ollama API. Retorna np.array (n, dim)."""
    embs = []
    for t in textos:
        r = requests.post(
            f'{OLLAMA_BASE_URL}/api/embeddings',
            json={'model': modelo, 'prompt': t},
            timeout=60
        )
        r.raise_for_status()
        embs.append(r.json()['embedding'])
    arr = np.array(embs, dtype='float32')
    # Normalizar L2 (necessário para FAISS IndexFlatIP)
    arr = arr / (np.linalg.norm(arr, axis=1, keepdims=True) + 1e-12)
    return arr


def encode_st(textos, modelo_st):
    """Encoda lista de textos com sentence-transformers já carregado."""
    arr = modelo_st.encode(textos, convert_to_numpy=True,
                            normalize_embeddings=True, show_progress_bar=False)
    return arr.astype('float32')


print('✅ Funções de encoding definidas')


In [ ]:
# PASSO 2.2 — Lista de modelos a testar (filtrada por disponibilidade)

modelos_a_testar = []

# Ollama: nomic-embed-text
if ollama_status['online'] and ollama_status['tem_nomic']:
    modelos_a_testar.append({
        'nome': 'nomic-embed-text (Ollama)',
        'tipo': 'ollama',
        'modelo': 'nomic-embed-text',
        'dim_esperada': 768,
    })

# Ollama: mxbai-embed-large (opcional)
if ollama_status['online'] and ollama_status['tem_mxbai']:
    modelos_a_testar.append({
        'nome': 'mxbai-embed-large (Ollama)',
        'tipo': 'ollama',
        'modelo': 'mxbai-embed-large',
        'dim_esperada': 1024,
    })

# Local: BGE-M3
modelos_a_testar.append({
    'nome': 'BGE-M3 (local)',
    'tipo': 'local',
    'modelo': 'BAAI/bge-m3',
    'dim_esperada': 1024,
})

# Local: MiniLM
modelos_a_testar.append({
    'nome': 'MiniLM-Multilingual (local)',
    'tipo': 'local',
    'modelo': 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2',
    'dim_esperada': 768,
})

print(f'🧪 {len(modelos_a_testar)} modelos serão testados:')
for m in modelos_a_testar:
    print(f'   • {m["nome"]:35s}  [{m["tipo"]:6s}]  dim={m["dim_esperada"]}')

if len(modelos_a_testar) < 2:
    print('\n⚠️  Menos de 2 modelos disponíveis — comparação ficará limitada.')


In [ ]:
# PASSO 2.3 — Encodar corpus com cada modelo, cronologicamente

# Preparar textos do corpus (titulo + conteudo)
textos_corpus = [f"{doc['titulo']}. {doc['texto']}" for doc in corpus]
textos_queries = [q['query'] for q in queries_avaliacao]

resultados = {}  # {nome_modelo: {emb_corpus, emb_queries, tempo, dim, ...}}

# Pré-carregar modelos locais (sentence-transformers)
from sentence_transformers import SentenceTransformer
modelos_st_carregados = {}

for m in modelos_a_testar:
    nome = m['nome']
    print(f'\n{"━"*60}')
    print(f'  🔢 ENCODING: {nome}')
    print(f'{"━"*60}')

    try:
        if m['tipo'] == 'ollama':
            # Aquecimento (primeiro request costuma ser mais lento)
            _ = encode_ollama([textos_corpus[0]], m['modelo'])

            inicio = time.time()
            emb_corpus = encode_ollama(textos_corpus, m['modelo'])
            t_corpus = time.time() - inicio

            inicio = time.time()
            emb_queries = encode_ollama(textos_queries, m['modelo'])
            t_queries = time.time() - inicio

        else:
            if m['modelo'] not in modelos_st_carregados:
                print(f'   ↓ Carregando {m["modelo"]} (pode demorar na 1ª vez)...')
                modelos_st_carregados[m['modelo']] = SentenceTransformer(m['modelo'], device=DEVICE)

            mod = modelos_st_carregados[m['modelo']]

            inicio = time.time()
            emb_corpus = encode_st(textos_corpus, mod)
            t_corpus = time.time() - inicio

            inicio = time.time()
            emb_queries = encode_st(textos_queries, mod)
            t_queries = time.time() - inicio

        resultados[nome] = {
            'emb_corpus': emb_corpus,
            'emb_queries': emb_queries,
            't_corpus': t_corpus,
            't_queries': t_queries,
            'dim': emb_corpus.shape[1],
            'tipo': m['tipo'],
            'modelo_id': m['modelo'],
        }

        print(f'   ✅ Corpus  ({len(textos_corpus)} docs):    {t_corpus:.2f}s   '
              f'→ {t_corpus/len(textos_corpus)*1000:.0f} ms/doc')
        print(f'   ✅ Queries ({len(textos_queries)} queries): {t_queries:.2f}s')
        print(f'   📐 Dim: {emb_corpus.shape[1]}')

    except Exception as e:
        print(f'   ❌ Erro: {e}')
        print(f'   → Modelo será excluído da comparação')

print(f'\n🎯 {len(resultados)} modelos encodados com sucesso')


In [ ]:
# PASSO 2.4 — Resumo do encoding (timing comparativo)

if not resultados:
    raise RuntimeError('Nenhum modelo foi encodado — verifique Ollama/dependências')

df_timing = pd.DataFrame([
    {
        'Modelo': nome,
        'Dim': r['dim'],
        'Tempo corpus (s)': round(r['t_corpus'], 2),
        'ms/doc': int(r['t_corpus'] / len(textos_corpus) * 1000),
        'Tempo queries (s)': round(r['t_queries'], 2),
        'Throughput (docs/s)': round(len(textos_corpus) / r['t_corpus'], 1),
    }
    for nome, r in resultados.items()
])
df_timing = df_timing.sort_values('Tempo corpus (s)').reset_index(drop=True)

print('📊 RESUMO DO ENCODING')
print('=' * 70)
print(df_timing.to_string(index=False))

# Gráfico
fig, ax = plt.subplots(figsize=(12, 5))
nomes = df_timing['Modelo'].tolist()
tempos = df_timing['Tempo corpus (s)'].tolist()
cores = [CORES_MODELOS.get(n, '#888') for n in nomes]
bars = ax.barh(nomes, tempos, color=cores, edgecolor='white')
for bar, val in zip(bars, tempos):
    ax.text(val + 0.05, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}s', va='center', fontweight='bold')
ax.set_xlabel('Tempo de encoding do corpus (s)')
ax.set_title('Velocidade de Encoding por Modelo', fontweight='bold')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()


---

# 📊 FASE 3 — Avaliação Quantitativa

Para cada modelo, calcularemos:
- **Hit Rate@K** (K∈{1,3,5}) — proporção de queries com ≥1 doc relevante no top-K
- **Recall@K** — fração de relevantes recuperados no top-K
- **MRR** — Mean Reciprocal Rank
- **NDCG@5 e NDCG@10** — ranking com relevância graduada (0-2)
- **AUC-ROC** — separabilidade entre similaridades positivas e negativas

> Este é o mesmo framework metodológico construído na Parte 2 do LAB2.


In [ ]:
# PASSO 3.1 — Funções de métricas (reaproveitadas do LAB2)

def hit_rate_at_k(sims, relev, k):
    n_q = sims.shape[0]
    hits = 0
    for i in range(n_q):
        topk = np.argsort(sims[i])[::-1][:k]
        if (relev[i, topk] >= 1).any():
            hits += 1
    return hits / n_q


def recall_at_k(sims, relev, k):
    n_q = sims.shape[0]
    recalls = []
    for i in range(n_q):
        rels_tot = (relev[i] >= 1).sum()
        if rels_tot == 0:
            continue
        topk = np.argsort(sims[i])[::-1][:k]
        recalls.append((relev[i, topk] >= 1).sum() / rels_tot)
    return float(np.mean(recalls))


def mrr(sims, relev):
    n_q = sims.shape[0]
    rrs = []
    for i in range(n_q):
        ranking = np.argsort(sims[i])[::-1]
        for rank, idx in enumerate(ranking, 1):
            if relev[i, idx] >= 1:
                rrs.append(1.0 / rank)
                break
        else:
            rrs.append(0.0)
    return float(np.mean(rrs))


print('✅ Funções de métricas definidas')


In [ ]:
# PASSO 3.2 — Calcular todas as métricas para cada modelo

metricas_por_modelo = {}

for nome, r in resultados.items():
    sims = cosine_similarity(r['emb_queries'], r['emb_corpus'])
    r['sims_q_d'] = sims  # guardar para uso depois (Fase 4 e 5)

    m = {}
    # Hit Rate e Recall
    for k in [1, 3, 5]:
        m[f'Hit@{k}']    = hit_rate_at_k(sims, relevancias, k)
        m[f'Recall@{k}'] = recall_at_k(sims, relevancias, k)
    # MRR
    m['MRR'] = mrr(sims, relevancias)
    # NDCG
    m['NDCG@5']  = float(ndcg_score(relevancias, sims, k=5))
    m['NDCG@10'] = float(ndcg_score(relevancias, sims, k=10))

    # AUC-ROC (separabilidade pos/neg)
    pos = sims[relevancias >= 1]
    neg = sims[relevancias == 0]
    y_true = np.concatenate([np.ones_like(pos), np.zeros_like(neg)])
    y_score = np.concatenate([pos, neg])
    m['AUC-ROC'] = float(roc_auc_score(y_true, y_score))
    m['Δμ pos-neg'] = float(pos.mean() - neg.mean())

    metricas_por_modelo[nome] = m

df_metricas = pd.DataFrame(metricas_por_modelo).T
print('📊 MÉTRICAS POR MODELO')
print('=' * 100)
print(df_metricas.round(4).to_string())


In [ ]:
# PASSO 3.3 — Visualização: radar chart comparativo

metricas_radar = ['Hit@1', 'Hit@3', 'Recall@5', 'MRR', 'NDCG@5', 'NDCG@10', 'AUC-ROC']

fig, ax = plt.subplots(figsize=(11, 11), subplot_kw=dict(polar=True))

angles = np.linspace(0, 2*np.pi, len(metricas_radar), endpoint=False).tolist()
angles += angles[:1]

for nome in metricas_por_modelo:
    valores = [df_metricas.loc[nome, m] for m in metricas_radar]
    valores += valores[:1]
    cor = CORES_MODELOS.get(nome, '#666')
    ax.plot(angles, valores, color=cor, linewidth=2.5, label=nome)
    ax.fill(angles, valores, color=cor, alpha=0.15)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metricas_radar, fontsize=11)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_title('Radar Chart — Comparação Multimétrica de Embeddings',
             fontsize=15, fontweight='bold', pad=24)
ax.legend(loc='upper right', bbox_to_anchor=(1.32, 1.08), fontsize=10)
ax.grid(True, alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:
# PASSO 3.4 — Heatmap consolidado

fig, ax = plt.subplots(figsize=(14, max(3, 1 + len(metricas_por_modelo))))
data = df_metricas.values
im = ax.imshow(data, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(len(df_metricas.columns)))
ax.set_xticklabels(df_metricas.columns, rotation=30, ha='right')
ax.set_yticks(range(len(df_metricas.index)))
ax.set_yticklabels(df_metricas.index)
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        ax.text(j, i, f'{data[i, j]:.3f}', ha='center', va='center',
                color='black', fontsize=10, fontweight='bold')
ax.set_title('Heatmap — Métricas por Modelo (verde = melhor)', fontweight='bold', pad=10)
plt.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
plt.tight_layout()
plt.show()

# Identificar vencedor por métrica e placar geral
print('\n🏆 VENCEDOR POR MÉTRICA')
print('=' * 70)
vitorias = {n: 0 for n in metricas_por_modelo}
for col in df_metricas.columns:
    valores = df_metricas[col].to_dict()
    vencedor = max(valores, key=valores.get)
    vitorias[vencedor] += 1
    print(f'  {col:15s} → {vencedor}')

print('\n📌 PLACAR FINAL DE QUALIDADE')
print('=' * 70)
for n, v in sorted(vitorias.items(), key=lambda x: -x[1]):
    print(f'   {n}: {v} vitórias')

vencedor_qualidade = max(vitorias, key=vitorias.get)
print(f'\n🥇 Vencedor em qualidade de retrieval: {vencedor_qualidade}')


---

# 🤖 FASE 4 — RAG End-to-End (Retrieval + Geração)

Agora vamos testar cada modelo em um **pipeline RAG completo**:
1. Recebe uma query jurídica em linguagem natural
2. Faz retrieval (top-3) usando o modelo de embedding
3. Monta o contexto com os 3 docs recuperados
4. Envia para o LLM `llama3.2:3b` (via Ollama) e captura a resposta
5. Mede latência e qualidade (categoria correta no top-1)

Vamos usar **a mesma query** para todos os modelos para isolar a variável "qualidade do embedding".


In [ ]:
# PASSO 4.1 — Construir índices FAISS por modelo

indices_faiss = {}
for nome, r in resultados.items():
    dim = r['emb_corpus'].shape[1]
    idx = faiss.IndexFlatIP(dim)  # produto interno = cosseno (vetores já normalizados)
    idx.add(r['emb_corpus'])
    indices_faiss[nome] = idx
    print(f'  ✅ {nome:35s}  índice FAISS dim={dim}')


In [ ]:
# PASSO 4.2 — Função RAG genérica (parametrizada por modelo)

SYSTEM_RAG = (
    'Você é um assistente jurídico especializado em direito penal brasileiro. '
    'Responda APENAS com base nos documentos fornecidos abaixo. '
    'Se a informação não estiver nos documentos, diga explicitamente que não consta. '
    'Cite o número/título do documento ao fundamentar. '
    'NÃO invente artigos, precedentes ou fatos não presentes nos documentos.'
)


def rag_pipeline(query: str, nome_modelo: str, top_k: int = 3, llm_model: str = OLLAMA_LLM_MODEL):
    """Executa o pipeline RAG completo para um modelo de embedding específico.

    Retorna dict com: docs_recuperados, resposta, tempos, tokens, etc.
    """
    r = resultados[nome_modelo]
    out = {'modelo': nome_modelo, 'query': query, 'top_k': top_k, 'llm': llm_model}

    # 1) Encodar query
    t0 = time.time()
    if r['tipo'] == 'ollama':
        q_emb = encode_ollama([query], r['modelo_id'])
    else:
        q_emb = encode_st([query], modelos_st_carregados[r['modelo_id']])
    out['t_embed'] = time.time() - t0

    # 2) Retrieval via FAISS
    t0 = time.time()
    idx = indices_faiss[nome_modelo]
    scores, ids = idx.search(q_emb, top_k)
    out['t_retrieval'] = time.time() - t0
    out['docs_recuperados'] = [
        {'doc': corpus[int(i)], 'score': float(s)}
        for i, s in zip(ids[0], scores[0])
    ]

    # 3) Montar prompt
    contexto = '\n\n'.join([
        f'[Doc {i+1}] {d["doc"]["titulo"]}\nCategoria: {d["doc"]["categoria"]}\n{d["doc"]["texto"]}'
        for i, d in enumerate(out['docs_recuperados'])
    ])
    user_msg = f'DOCUMENTOS:\n{contexto}\n\nPERGUNTA: {query}\n\nResposta fundamentada:'
    out['prompt_user'] = user_msg

    # 4) Geração
    t0 = time.time()
    try:
        resp = client_llm.chat.completions.create(
            model=llm_model,
            messages=[
                {'role': 'system', 'content': SYSTEM_RAG},
                {'role': 'user',   'content': user_msg},
            ],
            temperature=0.2,
            max_tokens=400,
        )
        out['resposta'] = resp.choices[0].message.content
        out['tokens_input'] = resp.usage.prompt_tokens
        out['tokens_output'] = resp.usage.completion_tokens
    except Exception as e:
        out['resposta'] = f'[ERRO LLM: {e}]'
        out['tokens_input'] = 0
        out['tokens_output'] = 0
    out['t_geracao'] = time.time() - t0

    out['t_total'] = out['t_embed'] + out['t_retrieval'] + out['t_geracao']
    return out


print('✅ Pipeline RAG genérico pronto')


In [ ]:
# PASSO 4.3 — Executar RAG para uma query jurídica de demonstração com cada modelo

QUERY_DEMO = 'Quais são os requisitos para decretação de prisão preventiva?'

execucoes_rag = {}

if ollama_status['tem_llm']:
    print(f'🤖 QUERY DE TESTE: "{QUERY_DEMO}"')
    print(f'   LLM: {OLLAMA_LLM_MODEL}\n')

    for nome in resultados:
        print(f'━' * 70)
        print(f'  🧪 Pipeline RAG com embedding: {nome}')
        print(f'━' * 70)
        out = rag_pipeline(QUERY_DEMO, nome, top_k=3)
        execucoes_rag[nome] = out

        # Mostrar docs recuperados
        print('  📚 Top-3 recuperados:')
        for i, d in enumerate(out['docs_recuperados'], 1):
            print(f'     {i}. [{d["score"]:.3f}] [{d["doc"]["categoria"]:20s}] {d["doc"]["titulo"][:55]}')

        # Mostrar resposta
        print(f'\n  💬 Resposta:')
        for linha in out['resposta'].split('\n'):
            print(f'     {linha}')

        # Tempos
        print(f'\n  ⏱️  Embed: {out["t_embed"]*1000:.0f}ms | '
              f'Retrieval: {out["t_retrieval"]*1000:.0f}ms | '
              f'Geração: {out["t_geracao"]:.1f}s | '
              f'Total: {out["t_total"]:.1f}s')
        print(f'  🎫 Tokens: {out["tokens_input"]} in / {out["tokens_output"]} out\n')
else:
    print('⚠️  LLM (llama3.2:3b) não disponível no Ollama — pulando Fase 4')
    print('    Para executar: ollama pull llama3.2:3b')


In [ ]:
# PASSO 4.4 — Comparação tabular das execuções RAG

if execucoes_rag:
    df_rag = pd.DataFrame([
        {
            'Modelo': nome,
            'Top-1 doc': out['docs_recuperados'][0]['doc']['titulo'][:40],
            'Cat. top-1': out['docs_recuperados'][0]['doc']['categoria'],
            'Score top-1': round(out['docs_recuperados'][0]['score'], 3),
            'Embed (ms)': int(out['t_embed'] * 1000),
            'Retrieval (ms)': int(out['t_retrieval'] * 1000),
            'Geração (s)': round(out['t_geracao'], 1),
            'Total (s)': round(out['t_total'], 1),
            'Tokens out': out['tokens_output'],
        }
        for nome, out in execucoes_rag.items()
    ])

    print('📊 COMPARAÇÃO DAS EXECUÇÕES RAG')
    print('=' * 100)
    print(df_rag.to_string(index=False))

    # Gráfico de tempos comparativos
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    nomes = list(execucoes_rag.keys())
    cores = [CORES_MODELOS.get(n, '#888') for n in nomes]

    # Tempo total
    ax = axes[0]
    totais = [execucoes_rag[n]['t_total'] for n in nomes]
    ax.barh(nomes, totais, color=cores, edgecolor='white')
    for i, v in enumerate(totais):
        ax.text(v + 0.05, i, f'{v:.1f}s', va='center', fontweight='bold')
    ax.set_xlabel('Tempo total do pipeline (s)')
    ax.set_title('Latência End-to-End por Modelo de Embedding', fontweight='bold')
    ax.invert_yaxis()
    ax.grid(axis='x', alpha=0.3)

    # Stacked: embed + retrieval + geração
    ax = axes[1]
    t_embed = [execucoes_rag[n]['t_embed'] for n in nomes]
    t_retr  = [execucoes_rag[n]['t_retrieval'] for n in nomes]
    t_gen   = [execucoes_rag[n]['t_geracao'] for n in nomes]
    y = np.arange(len(nomes))
    ax.barh(y, t_embed, color='#3498DB', label='Embed query')
    ax.barh(y, t_retr,  left=t_embed, color='#27AE60', label='Retrieval FAISS')
    ax.barh(y, t_gen,   left=[a+b for a,b in zip(t_embed, t_retr)],
            color='#E74C3C', label='Geração LLM')
    ax.set_yticks(y)
    ax.set_yticklabels(nomes)
    ax.set_xlabel('Tempo (s)')
    ax.set_title('Composição do Tempo Total', fontweight='bold')
    ax.invert_yaxis()
    ax.legend(loc='lower right')
    ax.grid(axis='x', alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print('⚠️  Sem execuções RAG para comparar (LLM offline)')


---

# 📡 FASE 5 — Observabilidade com LangFuse

Vamos registrar **um trace estruturado por execução RAG** no LangFuse, contendo:
- **Trace** (raiz): `pipeline-rag-comparativo` com metadados do modelo
- **Span 1** — `embedding-query` (qual modelo, tempo, dim)
- **Span 2** — `faiss-retrieval` (top-K, scores, categorias recuperadas)
- **Generation** — `ollama-llm` (modelo, prompt, resposta, tokens)
- **Score** — `categoria-top1-correta` (1.0 se acertou, 0.0 caso contrário)

Após executar, abra `http://localhost:3000` (LangFuse local) e use a sessão_id comum para **comparar lado a lado** os 4 pipelines no dashboard.


In [ ]:
# PASSO 5.1 — Inicializar LangFuse e enviar traces (1 por modelo)

if LANGFUSE_OK and execucoes_rag:
    from langfuse import Langfuse

    langfuse = Langfuse(
        public_key=LANGFUSE_PUBLIC_KEY,
        secret_key=LANGFUSE_SECRET_KEY,
        host=LANGFUSE_HOST,
    )

    # session_id comum aos 4 traces → permite agrupá-los no dashboard
    session_id = str(uuid.uuid4())
    print(f'📡 Enviando traces ao LangFuse')
    print(f'   Host:       {LANGFUSE_HOST}')
    print(f'   Session ID: {session_id}\n')

    traces_criados = []

    for nome, out in execucoes_rag.items():
        # Categoria-alvo conhecida? (se a query DEMO for sobre prisão preventiva → Direito Processual)
        # Aqui inferimos pelo conteúdo da query — para o demo é "Direito Processual"
        categoria_esperada = 'Direito Processual'  # heurística para a QUERY_DEMO
        categoria_obtida = out['docs_recuperados'][0]['doc']['categoria']
        acertou = 1.0 if categoria_obtida == categoria_esperada else 0.0

        # Tipo de embedding (ollama vs local)
        tipo_embed = resultados[nome]['tipo']

        trace = langfuse.trace(
            name='pipeline-rag-comparativo',
            session_id=session_id,
            metadata={
                'modulo':            'MBA RAG — Aula 1 — Exemplo',
                'modelo_embedding':  nome,
                'tipo_embedding':    tipo_embed,
                'dim_embedding':     resultados[nome]['dim'],
                'llm_model':         OLLAMA_LLM_MODEL,
                'top_k':             3,
                'categoria_top1':    categoria_obtida,
                'categoria_esperada': categoria_esperada,
            },
            tags=['aula1', 'exemplo', 'comparativo-embeddings', tipo_embed],
            input=out['query'],
            output=out['resposta'],
        )

        # Span 1: embedding da query
        span_embed = trace.span(
            name='embedding-query',
            input=out['query'],
            metadata={
                'modelo': nome,
                'dim':    resultados[nome]['dim'],
                'tipo':   tipo_embed,
            },
        )
        span_embed.end(output={'duracao_ms': int(out['t_embed'] * 1000)})

        # Span 2: retrieval
        span_retr = trace.span(
            name='faiss-retrieval',
            metadata={'top_k': 3, 'duracao_ms': int(out['t_retrieval'] * 1000)},
        )
        span_retr.end(output=[
            {
                'rank': i+1,
                'doc_id': d['doc']['id'],
                'titulo': d['doc']['titulo'],
                'categoria': d['doc']['categoria'],
                'score': d['score'],
            }
            for i, d in enumerate(out['docs_recuperados'])
        ])

        # Generation: LLM
        gen = trace.generation(
            name='ollama-generation',
            model=OLLAMA_LLM_MODEL,
            model_parameters={'temperature': 0.2, 'max_tokens': 400},
            input=out['prompt_user'][:1500],
            output=out['resposta'],
            usage={
                'input':  out['tokens_input'],
                'output': out['tokens_output'],
            },
        )
        gen.end()

        # Score: categoria-top1-correta
        trace.score(
            name='categoria_top1_correta',
            value=acertou,
            comment=f'Esperado: {categoria_esperada} | Obtido: {categoria_obtida}',
        )

        traces_criados.append({'modelo': nome, 'trace_id': trace.id, 'acertou_categoria': acertou})
        print(f'  ✅ {nome:35s} → trace_id={trace.id[:8]}... (acertou cat: {bool(acertou)})')

    langfuse.flush()
    print(f'\n📊 {len(traces_criados)} traces enviados')
    print(f'   Acesse {LANGFUSE_HOST} e filtre por session_id = {session_id}')
    print(f'   Você verá os 4 pipelines lado a lado para comparação visual.')
else:
    if not LANGFUSE_OK:
        print('⚠️  LangFuse não configurado — pulando Fase 5')
        print('    Configure LANGFUSE_PUBLIC_KEY e LANGFUSE_SECRET_KEY no ~/mba-rag/.env')
    elif not execucoes_rag:
        print('⚠️  Sem execuções RAG (LLM offline) — pulando Fase 5')


### 💡 Análise no Dashboard LangFuse

Ao abrir o LangFuse com o `session_id` capturado acima, você verá:

| Visão | O que comparar |
|-------|----------------|
| **Trace list** | Os 4 pipelines agrupados pela sessão — comparação imediata de tempo e tokens |
| **Span breakdown** | Quanto tempo o embedding consumiu em cada modelo (Ollama HTTP geralmente é mais lento) |
| **Generation details** | Mesmo LLM gerou respostas diferentes? Em que medida o contexto recuperado mudou? |
| **Scores** | Quantos dos 4 modelos acertaram a categoria-alvo? |

Os traces ficam persistidos no LangFuse local — você pode voltar e analisar depois, ou compartilhar com colegas.


---

# 🎓 FASE 6 — Parecer Técnico Final

Vamos consolidar **TUDO** que medimos (qualidade, velocidade, dependências) em um **parecer técnico fundamentado** que o aluno pode usar como referência para a escolha do modelo em projetos jurídicos reais.

Critérios de decisão considerados:
1. **Qualidade de retrieval** (Hit@K, MRR, NDCG, AUC) — Fase 3
2. **Latência operacional** (encoding por doc, RAG end-to-end) — Fases 2 e 4
3. **Dependências infraestruturais** (precisa de Ollama? GPU? rede?)
4. **Custo** (todos os 4 são gratuitos, mas variam em RAM/VRAM)
5. **Manutenibilidade** (versionamento, fine-tuning, atualizações)


In [ ]:
# PASSO 6.1 — Tabela multicritério consolidada

# Para cada modelo, calcular um score 0-1 normalizado em cada critério
def normalizar(valores, maior_eh_melhor=True):
    arr = np.array(valores, dtype=float)
    if arr.max() == arr.min():
        return np.ones_like(arr)
    norm = (arr - arr.min()) / (arr.max() - arr.min())
    return norm if maior_eh_melhor else 1 - norm


# Coletar dados
nomes = list(metricas_por_modelo.keys())

# Critério 1: Qualidade média (média das 7 métricas radar)
qualidades = [
    np.mean([metricas_por_modelo[n][m] for m in ['Hit@1', 'Hit@3', 'Recall@5', 'MRR', 'NDCG@5', 'NDCG@10', 'AUC-ROC']])
    for n in nomes
]

# Critério 2: Velocidade (inverso do tempo de encoding por doc)
tempos = [resultados[n]['t_corpus'] / len(corpus) for n in nomes]
velocidades_norm = normalizar(tempos, maior_eh_melhor=False)

# Critério 3: Independência de serviços externos (locais > Ollama)
independencias = [1.0 if resultados[n]['tipo'] == 'local' else 0.5 for n in nomes]

# Critério 4: Modernidade / SOTA (heurística)
modernidade_map = {
    'nomic-embed-text (Ollama)':     0.8,
    'mxbai-embed-large (Ollama)':    0.85,
    'BGE-M3 (local)':                1.0,   # SOTA 2024 multilíngue
    'MiniLM-Multilingual (local)':   0.6,   # 2019, mais antigo
}
modernidades = [modernidade_map.get(n, 0.5) for n in nomes]

# Score composto (pesos típicos para projeto jurídico)
PESOS = {'qualidade': 0.40, 'velocidade': 0.20, 'independencia': 0.20, 'modernidade': 0.20}

scores_compostos = []
for i, n in enumerate(nomes):
    score = (
        PESOS['qualidade']    * qualidades[i] +
        PESOS['velocidade']   * velocidades_norm[i] +
        PESOS['independencia']* independencias[i] +
        PESOS['modernidade']  * modernidades[i]
    )
    scores_compostos.append(score)

df_parecer = pd.DataFrame({
    'Modelo': nomes,
    'Qualidade (média radar)': [round(q, 3) for q in qualidades],
    'Velocidade (norm)':       [round(v, 3) for v in velocidades_norm],
    'Independência':           independencias,
    'Modernidade':             modernidades,
    'SCORE COMPOSTO':          [round(s, 3) for s in scores_compostos],
}).sort_values('SCORE COMPOSTO', ascending=False).reset_index(drop=True)
df_parecer['Rank'] = df_parecer.index + 1

print('📊 PARECER TÉCNICO — TABELA MULTICRITÉRIO')
print('=' * 100)
print(df_parecer.to_string(index=False))

print(f'\n📐 Pesos aplicados: {PESOS}')


In [ ]:
# PASSO 6.2 — Gráfico do parecer (score composto + breakdown)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Painel 1: score composto
ax = axes[0]
ordem = df_parecer['Modelo'].tolist()
scores = df_parecer['SCORE COMPOSTO'].tolist()
cores = [CORES_MODELOS.get(n, '#888') for n in ordem]
bars = ax.barh(ordem, scores, color=cores, edgecolor='white')
for bar, val in zip(bars, scores):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontweight='bold', fontsize=11)
ax.set_xlabel('Score Composto (ponderado)')
ax.set_title('Ranking Final dos Modelos', fontsize=14, fontweight='bold')
ax.set_xlim(0, 1.05)
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)

# Painel 2: breakdown empilhado
ax = axes[1]
df_b = df_parecer.set_index('Modelo')
categorias_breakdown = ['Qualidade (média radar)', 'Velocidade (norm)', 'Independência', 'Modernidade']
pesos_vetor = [PESOS['qualidade'], PESOS['velocidade'], PESOS['independencia'], PESOS['modernidade']]
cores_cat = ['#27AE60', '#3498DB', '#9B59B6', '#F39C12']

bottom = np.zeros(len(df_b))
for cat, peso, c in zip(categorias_breakdown, pesos_vetor, cores_cat):
    contrib = df_b[cat].values * peso
    ax.barh(df_b.index, contrib, left=bottom, color=c, edgecolor='white',
            label=f'{cat} (peso {peso})')
    bottom += contrib
ax.set_xlabel('Contribuição ao score composto')
ax.set_title('Composição do Score por Critério', fontsize=14, fontweight='bold')
ax.invert_yaxis()
ax.legend(loc='lower right', fontsize=9)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

vencedor_final = df_parecer.iloc[0]['Modelo']
print(f'\n🥇 VENCEDOR FINAL: {vencedor_final}')
print(f'   Score composto: {df_parecer.iloc[0]["SCORE COMPOSTO"]:.3f}')


In [ ]:
# PASSO 6.3 — Parecer textual estruturado

print('═' * 90)
print('  PARECER TÉCNICO — SELEÇÃO DE MODELO DE EMBEDDING PARA RAG JURÍDICO')
print('  MBA em RAG & CAG Aplicados a Direito e Segurança Pública — Aula 1')
print('═' * 90)

print(f'\n📋 CONTEXTO TESTADO')
print(f'   • Corpus:        {len(corpus)} documentos jurídicos em {len(categorias_corpus)} categorias')
print(f'   • Queries:       {len(queries_avaliacao)} queries de avaliação (escala 0-2)')
print(f'   • Modelos:       {len(resultados)} candidatos')
print(f'   • LLM (geração): {OLLAMA_LLM_MODEL}')
print(f'   • Dispositivo:   {DEVICE}')

print(f'\n🏆 RANKING FINAL')
for i, row in df_parecer.iterrows():
    medalha = ['🥇', '🥈', '🥉', '🏅'][min(i, 3)]
    print(f'   {medalha} {row["Modelo"]:35s}  score = {row["SCORE COMPOSTO"]:.3f}')

vencedor = df_parecer.iloc[0]['Modelo']
print(f'\n✅ RECOMENDAÇÃO PRINCIPAL: {vencedor}')

# Recomendações específicas por cenário
print('\n📌 RECOMENDAÇÕES POR CENÁRIO DE USO JURÍDICO\n')

cenarios = [
    {
        'titulo': '1️⃣  PROTÓTIPO / POC / SALA DE AULA',
        'criterios': 'Velocidade de setup, sem dependência de infraestrutura externa',
        'recomenda': 'nomic-embed-text (Ollama) OU MiniLM-Multilingual (local)',
        'justifica': 'Ollama oferece API estável e modelo já instalado; MiniLM é o mais leve para CPU.'
    },
    {
        'titulo': '2️⃣  CORPUS JURÍDICO MÉDIO (< 50k documentos)',
        'criterios': 'Equilíbrio entre qualidade e custo operacional',
        'recomenda': 'BGE-M3 (local) — modelo SOTA multilíngue',
        'justifica': 'Melhor qualidade semântica em português, 8192 tokens, suporta busca densa+esparsa+multi-vetor.'
    },
    {
        'titulo': '3️⃣  PRODUÇÃO COM CORPUS GRANDE (> 100k docs) E SLA',
        'criterios': 'Qualidade + latência previsível + observabilidade',
        'recomenda': 'BGE-M3 servido via OpenSearch Neural Search Plugin (Aula 4)',
        'justifica': 'Combina qualidade SOTA com infraestrutura distribuída e versionamento.'
    },
    {
        'titulo': '4️⃣  AMBIENTE OFFLINE / AIR-GAPPED (TJ, MP, PF)',
        'criterios': 'Zero dependência de rede externa, controle de modelo',
        'recomenda': 'BGE-M3 (local via sentence-transformers)',
        'justifica': 'Modelo HuggingFace baixável uma vez; após isso, 100% offline.'
    },
    {
        'titulo': '5️⃣  PROJETO FINAL DO MBA (Aula 12)',
        'criterios': 'Pipeline production-ready, alinhado com técnicas avançadas',
        'recomenda': 'BGE-M3 + Reranker BGE-Reranker-v2-M3 (#T03 da Aula 3)',
        'justifica': 'Bi-encoder rápido para K=50 + cross-encoder fino para top-5 → estado da arte.'
    },
]

for c in cenarios:
    print(f'   {c["titulo"]}')
    print(f'      Critério-chave: {c["criterios"]}')
    print(f'      → Recomenda: {c["recomenda"]}')
    print(f'      Justificativa: {c["justifica"]}')
    print()

print('═' * 90)
print('  ⚠️  CONSIDERAÇÕES ADICIONAIS')
print('═' * 90)
print('''
  • Os resultados deste lab usam APENAS 10 documentos e 8 queries — para decisões
    de produção, repita a avaliação com corpus mínimo de 1000 documentos.

  • A análise qualitativa do output do LLM (Fase 4) revela quanto cada embedding
    influencia a RESPOSTA FINAL, não apenas o ranking de docs.

  • Os traces no LangFuse (Fase 5) servirão de baseline para comparar com técnicas
    avançadas (Reranker, HyDE, Multi-Query) a partir da Aula 3.

  • RAGAS (Aula 5) adicionará 4 métricas obrigatórias para validar a qualidade
    da resposta do LLM, não apenas do retrieval.
''')

print('═' * 90)
print(f'  ✅ Parecer concluído — {vencedor} foi o vencedor neste dataset jurídico')
print('═' * 90)


---

## ✅ Checklist do Exemplo

- [ ] Fase 0: ambiente verificado (Python, Ollama, LangFuse, GPU)
- [ ] Fase 1: corpus + 8 queries + matriz de relevância 0-2 construídos
- [ ] Fase 2: 2-4 modelos encodaram o corpus com timing registrado
- [ ] Fase 3: Hit@K, MRR, NDCG, AUC calculados → radar chart + heatmap gerados
- [ ] Fase 4: pipeline RAG executado (embed + retrieval + llama3.2:3b) para cada modelo
- [ ] Fase 5: traces enviados ao LangFuse (4 traces × spans estruturados)
- [ ] Fase 6: parecer técnico fundamentado com recomendação por cenário

## 🚀 Próximos Passos no MBA

1. **Aula 2 — Naive RAG (#T01)** — pipeline production-grade com chunking estruturado
2. **Aula 3 — Advanced RAG + Reranker (#T03)** — adicione BGE-Reranker-v2-M3 ao topo
3. **Aula 4 — OpenSearch Hybrid Search (#T04, #T09)** — substitua FAISS por OpenSearch
4. **Aula 5 — RAGAS** — métricas obrigatórias de qualidade do LLM (não só do retrieval)

## 🎯 Exercícios Sugeridos

### Exercício 1 — Ampliar o ground truth
Adicione 12 queries adicionais (total 20) cobrindo categorias subrepresentadas. Recompute todas as métricas — o ranking dos modelos mudou?

### Exercício 2 — Avaliação por categoria
Quebre as métricas por `categoria_alvo` (ex.: NDCG médio em Direito Processual vs Crimes Cibernéticos). Algum modelo é especialista em alguma categoria?

### Exercício 3 — Análise de hard negatives
Identifique os pares (query, doc) com **falsa similaridade alta** (sim ≥ 0.7 mas rel = 0). Esses são candidatos a fine-tuning. Liste os 5 piores.

### Exercício 4 — Trocar o LLM
Repita a Fase 4 e 5 com `mistral:7b-instruct` ou `qwen2.5:7b-instruct`. O LLM mascara ou amplifica as diferenças do embedding?

### Exercício 5 — Adicionar Reranker (preview Aula 3)
Após o retrieval top-10 com BGE-M3, aplique `BAAI/bge-reranker-v2-m3` para reordenar e medir o ganho em NDCG@3.
